# Feline Datasets 

From the genotypes with felines, create dataset for felines only in each genotype

In [1]:
# Housekeeping

import os
import pandas as pd

update_date = "6-5-2025"

home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
cat_files = downloads + "Cats/Datasets/"

os.chdir(home + "references/")
states_ref = pd.read_csv("states_ref.csv")

In [2]:
# Function to prepare dataframes
def fasta_df(file_name, state_ref):

    fasta = pd.DataFrame()
    headers = []
    isolate_ids = []
    isolate_names = []
    subtypes = []
    segments = []
    collection_dates = []
    sequences = []
    host_types = []
    species = []
    genotypes = []
    with open(file_name) as f:
        lines = f.readlines()
        for num, line in enumerate(lines):
            # print(line)
            if line[0] == ">": # If it's a header
                if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
                    header = line[1:].strip() # Remove the ">"
                    # print(header)
                    split_header = header.split("|")
                    split_first_header = split_header[0].split("/")
                    # print(split_header)
                    headers.append(header) 
                    isolate_ids.append(split_first_header[3])
                    isolate_names.append(split_header[0]) # We'll need to extract data from this too
                    # print(split_header[2].split("_")[-1])
                    subtypes.append(split_header[1])  # Get only H5N1
                    genotypes.append(split_header[-1])
                    segments.append(file_name.split("_")[-3])
                    host_types.append(split_header[3])
                    species.append(split_first_header[1])
                    # if split_header[4] == "2024-01-01":
                    #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                    # elif split_header[4] == "2025-01-01":
                    #     collection_dates.append("2025")
                    # else: 
                    collection_dates.append(split_header[2].split("_")[-1])
                    if num < len(lines): # If we're not at the last line
                        # for i, l in enumerate(lines[num + 1:]):
                        i = num
                        sequence = ""
                        # print(lines[i])
                        # print(lines[i + 1])
                        while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                            sequence = sequence + lines[i + 1].strip()
                            i += 1
                        sequences.append(sequence) # Add next line to sequences
        f.close()

    # Create columns for data frame 
    fasta["Header"] = headers
    fasta["Isolate_Id"] = isolate_ids
    fasta["Isolate_Name"] = isolate_names
    fasta["Subtype"] = subtypes
    fasta["Segment"] = segments
    # Geo_Location is more complicated
    fasta["Geo_Location"] = fasta["Header"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    fasta["Date Collected"] = collection_dates
    fasta["Species"] = species
    fasta["Host_Type"] = host_types
    fasta["Genotype"] = genotypes
    fasta["Sequence"] = sequences
    
    return fasta

# Grab files
cat_genotypes = set()
dfs = {}
for dirpath, dirs, files in os.walk(downloads + "Combinations/GISAID_Andersen/11-01-2021--04-14-2025_all_genotypes-20250604T192435Z-1-001/"):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        with open(file_name) as f:
            lines = f.readlines()
            for line in lines:
                if "feline" in line:
                    print(file_name)
                    cat_genotypes.add(line.split("|")[-1]) # The genotype
                    dfs[file_name] = fasta_df(file_name, states_ref)
                    break 

C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Combinations/GISAID_Andersen/11-01-2021--04-14-2025_all_genotypes-20250604T192435Z-1-001/11-01-2021--04-14-2025_all_genotypes\A3_HA_combined_04-14-2025.fasta
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Combinations/GISAID_Andersen/11-01-2021--04-14-2025_all_genotypes-20250604T192435Z-1-001/11-01-2021--04-14-2025_all_genotypes\A3_MP_combined_04-14-2025.fasta
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Combinations/GISAID_Andersen/11-01-2021--04-14-2025_all_genotypes-20250604T192435Z-1-001/11-01-2021--04-14-2025_all_genotypes\A3_NA_combined_04-14-2025.fasta
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Combinations/GISAID_Andersen/11-01-2021--04-14-2025_all_genotypes-20250604T192435Z-1-001/11-01-2021--04-14-2025_all_genotypes\A3_NP_combined_04-14-2025.fasta
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Combinations/GISAID_Andersen/11-01-2021--04-14-2025_all_genotyp

In [ ]:
os.chdir(cat_files)

def df_to_fasta(fasta, file_name, output_path):

    output_file = open(output_path + file_name, "w")

    for index, row in fasta.iterrows():
        name = fasta.loc[index, "Header"]
        sequence = fasta.loc[index, "Sequence"]
    # First is header, second is sequence
        output_file.write(">" + name + "\n")
        output_file.write(sequence + "\n")
    output_file.close()

# Get cats only
cat_dfs = {}
for key in dfs.keys():
    df_value = dfs[key]
    cat_df = df_value[df_value["Host_Type"] == "feline"]
    cat_dfs[key] = cat_df
    title = key.split("\\")[-1]
    title = "cat_" + update_date + "_" + title
    print(title)
    print(cat_df)
    df_to_fasta(cat_df, title, cat_files + "cat_genotypes_11-01-2021--06-05-2025/")

cat_6-5-2025_A3_HA_combined_04-14-2025.fasta
                                                Header     Isolate_Id  \
108  A/cat/Minnesota/24-036601-001/2024|H5N1|2024-1...  24-036601-001   

                           Isolate_Name Subtype Segment Geo_Location  \
108  A/cat/Minnesota/24-036601-001/2024    H5N1      HA       USA-MN   

    Date Collected Species Host_Type Genotype  \
108     2024-12-06     cat    feline       A3   

                                              Sequence  
108  atggagaacatagtacttcttcttgcaataattagccttgttaaaa...  


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/6-5-2025/cat_6-5-2025_A3_HA_combined_04-14-2025.fasta'

# Concatenating files with cat genotypes

In [6]:
new_cat_files = cat_files + "GISAID_Andersen_NCBI_Virus/cat_genotypes_04-14-2025--06-05-2025/"
old_cat_files = cat_files + "GISAID_Andersen_NCBI_Virus/cat_genotypes_11-01-2021--04-14-2025/"
output_files = cat_files + "GISAID_Andersen_NCBI_Virus/cat_genotypes_11-01-2021--06-05-2025/"



In [7]:
old = {}
for dirpath, dirs, files in os.walk(old_cat_files):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        with open(file_name) as f:
            lines = f.readlines()
            all_lines = ""
            for line in lines:
                all_lines = all_lines + line
            # print(all_lines)
            
            old[file_name] = all_lines

new = {}
for dirpath, dirs, files in os.walk(new_cat_files):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        with open(file_name) as f:
            lines = f.readlines()
            all_lines = ""
            for line in lines:
                all_lines = all_lines + line
            # print(all_lines)
            
            new[file_name] = all_lines

# print(old["C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/GISAID_Andersen_NCBI_Virus/cat_genotypes_04-14-2025--06-05-2025/cat_genotypes_04-14-2025--06-05-2025_A3_HA_combined_04-14-2025--06-05-2025.fasta"])

In [8]:
print(new.keys())

dict_keys(['C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/GISAID_Andersen_NCBI_Virus/cat_genotypes_04-14-2025--06-05-2025/A3_HA_combined_04-14-2025--06-05-2025.fasta', 'C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/GISAID_Andersen_NCBI_Virus/cat_genotypes_04-14-2025--06-05-2025/A3_MP_combined_04-14-2025--06-05-2025.fasta', 'C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/GISAID_Andersen_NCBI_Virus/cat_genotypes_04-14-2025--06-05-2025/A3_NA_combined_04-14-2025--06-05-2025.fasta', 'C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/GISAID_Andersen_NCBI_Virus/cat_genotypes_04-14-2025--06-05-2025/A3_NP_combined_04-14-2025--06-05-2025.fasta', 'C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/GISAID_Andersen_NCBI_Virus/cat_genotypes_04-14-2025--06-05-2025/A3_NS_combined_04-14-2025--06-05-2025.fasta', 'C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Fil

In [9]:
# Concatenate

for file_name in old.keys():
    genotype_seg = file_name.split("_")[-4].split("/")[-1] + "_" + file_name.split("_")[-3]
    print("old", genotype_seg)
    for new_name in new.keys():
        genotype_seg_new = new_name.split("_")[-4].split("/")[-1] + "_" + new_name.split("_")[-3]
        print("new", genotype_seg_new)
        if genotype_seg == genotype_seg_new:
            print(genotype_seg, genotype_seg_new) 
            # Concatenate
            filenames = [file_name, new_name]
            with open(output_files + genotype_seg + "_11-1-2021--6-5-2025_concat.fasta", 'w') as outfile:
                for fname in filenames:
                    with open(fname) as infile:
                        for line in infile:
                            if "Minor98" not in line: # Was mistake in old dataset
                                outfile.write(line)


old A3_HA
new A3_HA
A3_HA A3_HA
new A3_MP
new A3_NA
new A3_NP
new A3_NS
new A3_PA
new A3_PB1
new A3_PB2
new B3.13_HA
new B3.13_MP
new B3.13_NA
new B3.13_NP
new B3.13_NS
new B3.13_PA
new B3.13_PB1
new B3.13_PB2
new B3.2_HA
new B3.2_MP
new B3.2_NA
new B3.2_NP
new B3.2_NS
new B3.2_PA
new B3.2_PB1
new B3.2_PB2
new B3.5_HA
new B3.5_MP
new B3.5_NA
new B3.5_NP
new B3.5_NS
new B3.5_PA
new B3.5_PB1
new B3.5_PB2
new B3.6_HA
new B3.6_MP
new B3.6_NA
new B3.6_NP
new B3.6_NS
new B3.6_PA
new B3.6_PB1
new B3.6_PB2
new B3.7_HA
new B3.7_MP
new B3.7_NA
new B3.7_NP
new B3.7_NS
new B3.7_PA
new B3.7_PB1
new B3.7_PB2
new D1.1_HA
new D1.1_MP
new D1.1_NA
new D1.1_NP
new D1.1_NS
new D1.1_PA
new D1.1_PB1
new D1.1_PB2
old A3_MP
new A3_HA
new A3_MP
A3_MP A3_MP
new A3_NA
new A3_NP
new A3_NS
new A3_PA
new A3_PB1
new A3_PB2
new B3.13_HA
new B3.13_MP
new B3.13_NA
new B3.13_NP
new B3.13_NS
new B3.13_PA
new B3.13_PB1
new B3.13_PB2
new B3.2_HA
new B3.2_MP
new B3.2_NA
new B3.2_NP
new B3.2_NS
new B3.2_PA
new B3.2_PB1
new B